<a href="https://colab.research.google.com/github/Berkaycetinn/log-analyzer/blob/main/0_2_parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q drain3

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyiceberg 0.12.0 requires cachetools>=5.5, but you have cachetools 4.2.1 which is incompatible.


In [5]:
!git clone --depth 1 https://github.com/logpai/loghub.git
!pip install -q drain3
!git clone --depth 1 https://github.com/logpai/loghub.git

import re
import pandas as pd

pattern = re.compile(
    r"^(?P<date>\d{4}-\d{2}-\d{2}) (?P<time>\d{2}:\d{2}:\d{2},\d{3}) "
    r"(?P<level>[A-Z]+) \[(?P<thread>[^\]]*)\] (?P<component>\S+): (?P<message>.*)$"
)

rows = []
with open("loghub/Hadoop/Hadoop_2k.log", encoding="utf-8") as f:
    for line in f:
        m = pattern.match(line.rstrip("\n"))
        if m:
            rows.append(m.groupdict())

df = pd.DataFrame(rows)
gt = pd.read_csv("loghub/Hadoop/Hadoop_2k.log_structured.csv")
print(len(df), len(gt))

fatal: destination path 'loghub' already exists and is not an empty directory.
fatal: destination path 'loghub' already exists and is not an empty directory.
2000 2000


In [12]:
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
from drain3.masking import MaskingInstruction

def build_miner(sim_th=0.4, depth=4):
    config = TemplateMinerConfig()
    config.drain_sim_th = sim_th
    config.drain_depth = depth
    config.masking_instructions = [
        MaskingInstruction(r"\b(?:application|appattempt|container|job|task|attempt)_[\w]+", "ID"),
        MaskingInstruction(r"blk_-?\d+(?:_\d+)?", "BLK"),
        MaskingInstruction(r"\d+\.\d+\.\d+\.\d+", "IP"),
        MaskingInstruction(r"-?\d+(?:ms|s)\b", "DUR"),
        MaskingInstruction(r"\b\d+\b", "NUM"),
    ]
    return TemplateMiner(config=config)

def run_drain(messages, **kw):
    miner = build_miner(**kw)
    ids = []
    for msg in messages:
        res = miner.add_log_message(msg)
        ids.append(res["cluster_id"])
    templates = {c.cluster_id: c.get_template() for c in miner.drain.clusters}
    return ids, templates

df["cluster_id"], templates = run_drain(df["message"])
df["template"] = df["cluster_id"].map(templates)

print("Drain3 template sayısı:", len(templates))
print("Gerçek template sayısı: 114")

Drain3 template sayısı: 99
Gerçek template sayısı: 114


In [7]:
gt = pd.read_csv("loghub/Hadoop/Hadoop_2k.log_structured.csv")
print(gt.columns.tolist())   # sütun adlarını görmek için

# Satır sırası aynı mı? (1.0 çıkmalı)
print("Mesaj eşleşmesi:", (df["message"].values == gt["Content"].values).mean())

def grouping_accuracy(gt_ids, pred_ids):
    d = pd.DataFrame({"gt": list(gt_ids), "pred": list(pred_ids)})
    correct = 0
    for _, g in d.groupby("gt"):
        if g["pred"].nunique() == 1:
            p = g["pred"].iloc[0]
            if (d["pred"] == p).sum() == len(g):
                correct += len(g)
    return correct / len(d)

ga = grouping_accuracy(gt["EventId"], df["cluster_id"])
print(f"Grouping Accuracy: {ga:.3f}")

['LineId', 'Date', 'Time', 'Level', 'Process', 'Component', 'Content', 'EventId', 'EventTemplate']
Mesaj eşleşmesi: 0.9265
Grouping Accuracy: 0.963


In [8]:
d = pd.DataFrame({"gt": gt["EventId"], "pred": df["cluster_id"], "msg": df["message"]})

# Drain3'ün birleştirdiği (tek kümede birden fazla gerçek template)
merged = d.groupby("pred")["gt"].nunique()
print("Birleştirilmiş küme sayısı:", (merged > 1).sum())

# Drain3'ün böldüğü (bir gerçek template birden fazla kümede)
split = d.groupby("gt")["pred"].nunique()
print("Bölünmüş gerçek template sayısı:", (split > 1).sum())

pd.set_option("display.max_colwidth", 120)

print("\nEn çok bölünen 5 gerçek template:")
for gid in split.sort_values(ascending=False).head(5).index:
    print(f"\n{gid} ({split[gid]} parçaya bölündü):")
    for t in df.loc[gt["EventId"] == gid, "template"].unique()[:3]:
        print("   ", t)

print("\nEn çok birleştirilen 5 küme:")
for pid in merged.sort_values(ascending=False).head(5).index:
    print(f"\nKüme {pid} ({merged[pid]} gerçek template birleşti):")
    print("   Drain3 template:", templates[pid])
    for gid in d.loc[d["pred"] == pid, "gt"].unique()[:4]:
        print("   gerçek:", gt.loc[gt["EventId"] == gid, "EventTemplate"].iloc[0])

Birleştirilmiş küme sayısı: 5
Bölünmüş gerçek template sayısı: 0

En çok bölünen 5 gerçek template:

E1 (1 parçaya bölündü):
    <NUM> failures on node MININT-FNANLI5.fareast.corp.microsoft.com

E10 (1 parçaya bölündü):
    Address change detected. Old: msra-sa<NUM>/<IP>:<NUM> New: msra-sa<NUM>:<NUM>

E100 (1 parçaya bölündü):
    Task succeeded with attempt <ID>

E101 (1 parçaya bölündü):
    Task: <ID> - exited : java.net.NoRouteToHostException: No Route to Host from MININT-FNANLI5/<IP> to msra-sa<NUM>:<NUM> failed on socket timeout exception: java.net.NoRouteToHostException: No route to host: no further information; For more details see: http://wiki.apache.org/hadoop/NoRouteToHost

E102 (1 parçaya bölündü):
    <ID> <*> Transitioned from <*> to <*>

En çok birleştirilen 5 küme:

Küme 47 (11 gerçek template birleşti):
   Drain3 template: <ID> <*> Transitioned from <*> to <*>
   gerçek: task_<*> Task Transitioned from NEW to SCHEDULED
   gerçek: attempt_<*> TaskAttempt Transitioned fr

In [9]:
for sim_th in [0.3, 0.4, 0.5, 0.6, 0.7]:
    ids, tpls = run_drain(df["message"], sim_th=sim_th)
    ga = grouping_accuracy(gt["EventId"], ids)
    print(f"sim_th={sim_th}  template={len(tpls):3d}  GA={ga:.3f}")

sim_th=0.3  template= 99  GA=0.963
sim_th=0.4  template= 99  GA=0.963
sim_th=0.5  template= 99  GA=0.963
sim_th=0.6  template=101  GA=0.963
sim_th=0.7  template=111  GA=0.960


In [10]:
a = df["message"].str.strip().str.split().str.join(" ")
b = gt["Content"].astype(str).str.strip().str.split().str.join(" ")
print("Boşluklar normalize edilince eşleşme:", (a.values == b.values).mean())

bad = df.index[a.values != b.values]
print("Eşleşmeyen satır:", len(bad))
for i in bad[:5]:
    print(i, "\n  bizim:", repr(df.loc[i, "message"])[:150],
             "\n  gt   :", repr(gt.loc[i, "Content"])[:150])

Boşluklar normalize edilince eşleşme: 1.0
Eşleşmeyen satır: 0


In [11]:
sub = d[d["pred"].isin([47, 17])].copy()
sub["level"] = df["level"]
tmap = gt.drop_duplicates("EventId").set_index("EventId")["EventTemplate"]
sub["gt_template"] = sub["gt"].map(tmap)
pd.set_option("display.max_colwidth", 90)
print(sub.groupby(["pred", "gt_template", "level"]).size().to_string())

pred  gt_template                                                                            level
17    job_<*>Job Transitioned from INITED to SETUP                                           INFO      1
      job_<*>Job Transitioned from NEW to INITED                                             INFO      1
      job_<*>Job Transitioned from SETUP to RUNNING                                          INFO      1
47    attempt_<*> TaskAttempt Transitioned from ASSIGNED to RUNNING                          INFO     10
      attempt_<*> TaskAttempt Transitioned from FAIL_CONTAINER_CLEANUP to FAIL_TASK_CLEANUP  INFO      2
      attempt_<*> TaskAttempt Transitioned from FAIL_TASK_CLEANUP to FAILED                  INFO      2
      attempt_<*> TaskAttempt Transitioned from NEW to UNASSIGNED                            INFO     14
      attempt_<*> TaskAttempt Transitioned from RUNNING to FAIL_CONTAINER_CLEANUP            INFO      2
      attempt_<*> TaskAttempt Transitioned from RUNNING to SU

In [13]:
errs = df[df["level"].isin(["ERROR", "WARN", "FATAL"])]
print("ERROR/WARN için küme sayısı:", errs["cluster_id"].nunique())
print(errs["template"].value_counts().head(12).to_string())

ERROR/WARN için küme sayısı: 12
template
Address change detected. Old: msra-sa-<NUM>/<IP>:<NUM> New: msra-sa-<NUM>:<NUM>                                                                                                                                                                                                                             476
Failed to renew lease for [DFSClient_NONMAPREDUCE_1537864556_1] for <NUM> seconds. Will retry shortly ...                                                                                                                                                                                                   326
ERROR IN CONTACTING RM.                                                                                                                                                                                                                                                                                     147
Task: <ID> - exited : java.net.NoRouteToHostExc

In [14]:
SUSP = re.compile(r"fail|exception|error|timeout|timed out|refused|no route|unreachable|killed|denied", re.I)

df["suspicious"] = df["message"].apply(lambda m: bool(SUSP.search(m)))
df["important"] = df["level"].isin(["ERROR", "WARN", "FATAL"]) | df["suspicious"]

hidden = df[(df["level"] == "INFO") & df["suspicious"]]
print("INFO seviyesinde ama şüpheli satır:", len(hidden))
print(hidden.groupby("template").size().to_string())

pd.set_option("display.max_colwidth", 110)
print(hidden[["time", "message"]].head(15).to_string())

# ERROR/WARN kümeleri saf mı? (içlerine INFO karışmış mı)
for cid in df.loc[df["level"].isin(["ERROR", "WARN"]), "cluster_id"].unique():
    lv = df.loc[df["cluster_id"] == cid, "level"].value_counts().to_dict()
    if len(lv) > 1:
        print("Karışık küme", cid, lv)
print("Toplam önemli satır:", df["important"].sum(), "/", len(df))

INFO seviyesinde ama şüpheli satır: 16
template
<ID> <*> Transitioned from <*> to <*>                                                                                                                                                                                                                                                                                      6
<NUM> failures on node MININT-FNANLI5.fareast.corp.microsoft.com                                                                                                                                                                                                                                                           2
Added <ID> to list of failed maps                                                                                                                                                                                                                                                                                          2
D

In [15]:
SUSP = re.compile(r"fail|exception|error|timeout|timed out|refused|no route|unreachable|killed|denied", re.I)
BENIGN = re.compile(r"maxTaskFailuresPerNode")

df["suspicious"] = df["message"].apply(lambda m: bool(SUSP.search(m)) and not BENIGN.search(m))
df["important"] = df["level"].isin(["ERROR", "WARN", "FATAL"]) | df["suspicious"]
print(df["important"].sum())

975
